# Data Engineering & Pipelines

**Consulting Update - BigDataCompany**
To ensure the highest data quality for our machine learning and graph algorithms, this notebook constructs a scalable data preparation pipeline using PySpark. 

For the **Mental Health Dataset**, we implement advanced Regex pattern matching to strip out noise (URLs, mentions, hashtags) that pollute NLP models, followed by Spark MLlib's Tokenizer and StopWordsRemover. 
For the **Twitter Network Dataset**, we standardize schemas and rename entity columns to `src` and `dst`, satisfying the strict schema requirements for Spark GraphFrames.

In [0]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, trim, when, to_timestamp
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# Initialize Spark Session
spark = SparkSession.builder.appName("Preprocessing_Pipelines").getOrCreate()

current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, ".."))

mental_health_path = f"file:{repo_root}/Datasets/mental_health.csv"
twitter_graph_path = f"file:{repo_root}/Datasets/twitter_graph.csv"

mental_health_df = spark.read.csv(mental_health_path, header=True, inferSchema=True, multiLine=True, escape='"')
twitter_graph_df = spark.read.csv(twitter_graph_path, header=True, inferSchema=True)

### Advanced Text Cleansing (Regex)
Social media text is highly unstructured. Before applying basic alphabetic filters, we must aggressively target and remove hyper-links, user mentions, and hashtag symbols to prevent the model from learning structural noise rather than semantic emotion.

In [0]:
# 1. Remove URLs (http/https)
clean_text_df = mental_health_df.withColumn("Text_Stage1", regexp_replace(col("Text"), r"http\S+|www\.\S+", ""))
# 2. Remove Mentions (@user)
clean_text_df = clean_text_df.withColumn("Text_Stage2", regexp_replace(col("Text_Stage1"), r"@\w+", ""))
# 3. Remove Hashtag symbols but keep the word (or remove entirely depending on domain choice)
clean_text_df = clean_text_df.withColumn("Text_Stage3", regexp_replace(col("Text_Stage2"), r"#", ""))
# 4. Keep only alphabetic characters and lowercase
clean_text_df = clean_text_df.withColumn("Cleaned_Text", lower(regexp_replace(col("Text_Stage3"), r"[^a-zA-Z\s]", "")))
# 5. Trim extra whitespaces
clean_text_df = clean_text_df.withColumn("Cleaned_Text", trim(regexp_replace(col("Cleaned_Text"), r"\s+", " ")))

# Drop intermediate columns to save memory
clean_text_df = clean_text_df.drop("Text_Stage1", "Text_Stage2", "Text_Stage3")

### NLP Pipeline: Tokenization & Stop Words
Leveraging Spark MLlib to break sentences into arrays of tokens and filtering out common english stop words (e.g., "the", "is", "in"), heavily reducing the dimensionality of the text for TF-IDF or Word2Vec.

In [0]:
# Tokenize text into words
tokenizer = Tokenizer(inputCol="Cleaned_Text", outputCol="Tokens")
tokenized_df = tokenizer.transform(clean_text_df)

# Remove Stop Words
remover = StopWordsRemover(inputCol="Tokens", outputCol="Filtered_Tokens")
final_text_df = remover.transform(tokenized_df)

final_text_df.select("Text", "Filtered_Tokens", "Emotion").show(5, truncate=50)

### GraphFrames Pre-requisites & Schema Standardization
GraphFrames requires explicitly named `src` and `dst` columns to generate the Edge DataFrame. We also enforce boolean logic and parse string timestamps into Spark `TimestampType` to enable time-window network analysis.

In [0]:
# Rename columns for GraphFrames compatibility
graph_prep_df = twitter_graph_df.withColumnRenamed("Source", "src") \
                                .withColumnRenamed("Destination", "dst")

# Drop invalid edges
graph_prep_df = graph_prep_df.dropna(subset=["src", "dst"])

# Parse Timestamp & Boolean Types
graph_prep_df = graph_prep_df.withColumn("Relationship Date (UTC)", to_timestamp(col("Relationship Date (UTC)")))
graph_prep_df = graph_prep_df.withColumn("Source Verified", when(col("Source Verified") == "TRUE", True).otherwise(False))
graph_prep_df = graph_prep_df.withColumn("Destination Verified", when(col("Destination Verified") == "TRUE", True).otherwise(False))

graph_prep_df.printSchema()
graph_prep_df.select("src", "dst", "Source Verified", "Relationship Date (UTC)").show(5)

### Final Data Engineering Step: Optimized Storage
Writing the processed DataFrames in Parquet format. Parquet is highly optimized for Spark operations, preserving schemas and providing columnar compression to accelerate machine learning read times.

In [0]:
# final_text_df.write.parquet("dbfs:/FileStore/big_data_project/cleaned_mental_health.parquet", mode="overwrite")
# graph_prep_df.write.parquet("dbfs:/FileStore/big_data_project/cleaned_twitter_graph.parquet", mode="overwrite")
print("\u2705 Advanced Preprocessing complete. Data ready for MLlib and GraphFrames integration.")